# Phase 6 - Final predictions and district ranking

This notebook drives the Phase 6 deliverables. It performs **no modelling of its own**:
the ratified Ridge B model (Configuration B, alpha = 100, 77 predictors, 628 districts)
was fixed in Phase 3; the code below fits it on all eligible districts, persists it,
and scores every district.

**Important:** the values produced here are in-sample fitted values, not a performance
estimate. Generalisation evidence remains the grouped nested cross-validation in
`evaluation/metrics/phase3_nested_cv_results.csv`.

Full documentation: `reports/ml_report/PHASE_6_PREDICTION_SYSTEM.md`.

In [ ]:
import subprocess, sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()

for step in ('src/prediction/final_model.py', 'src/prediction/build_predictions.py'):
    print('---', step)
    result = subprocess.run([sys.executable, step], cwd=ROOT, capture_output=True, text=True)
    print(result.stdout[-2000:] or result.stderr[-2000:])
    assert result.returncode == 0, f'{step} failed'

In [ ]:
import json
import pandas as pd

metadata = json.loads((ROOT / 'models/final/ridge_b_metadata.json').read_text())
predictions = pd.read_csv(ROOT / 'evaluation/predictions/final_predictions_ridge_b_628.csv')
ranking = pd.read_csv(ROOT / 'evaluation/predictions/district_ranking_ridge_b_628.csv')

print('model      ', metadata['model_id'], '| alpha', metadata['alpha'])
print('trained    ', metadata['trained_at_utc'], '| districts', metadata['n_training_districts'])
print('predictions', len(predictions), 'rows | unique keys', predictions['district_key'].is_unique)
print('range      %.3f to %.3f pp (mean %.3f)' % (
    predictions['predicted_deprivation_rate_pct'].min(),
    predictions['predicted_deprivation_rate_pct'].max(),
    predictions['predicted_deprivation_rate_pct'].mean()))
print('flagged    ', predictions['physical_range_flag'].value_counts().to_dict())
ranking.head(10)[['rank','district_name','state_name','predicted_deprivation_rate_pct','relative_band']]

## Reading the output

- `predicted_deprivation_rate_pct` is the predicted **share of census houses classified
  Dilapidated** in the district (Census 2011), in percentage points.
- `physical_range_flag` marks the two districts whose fitted value falls below 0.
  Linear models can extrapolate outside the admissible range; values are reported as
  computed and are not clipped.
- `relative_band` is a **relative predicted quintile** (B1 lowest ... B5 highest), not a
  validated risk class. No external severity threshold exists for this outcome.
- 12 of the 640 districts have no D5 household denominator and are absent by design.
- The model under-predicts the most dilapidated districts by roughly 4.5 pp out of fold;
  the top of this ranking is directionally useful, not calibrated.